In [14]:
import pandas as pd
import os
print("Current working directory:", os.getcwd())

Current working directory: c:\Users\jhigh\Projects\triathlon-db\Mixed_Relay


In [ ]:
# 1. Read all sheets into a dict of DataFrames
print("Loading Mixed Relay data from Excel file...")
try:
    file_path = "USAT_MixedRelay_vJH.xlsx"
    sheets = pd.read_excel(file_path, sheet_name=None, header=0)    
except FileNotFoundError:
    raise FileNotFoundError(f"File '{file_path}' not found. Please check the path and try again.")  

# Remove 'Sheet1' if it exists
sheets.pop("Sheet1", None)



Loading Mixed Relay data from Excel file...


KeyError: 'Individual Race'

In [20]:
# 2. Prepare a list to collect cleaned DataFrames
cleaned = []

# 3. Define the shortlist and tier mapping
short_list = [
    "Chase McQueen", "Morgan Pearson", "John Reed", 
    "Reese Vannerson", "Sullivan Middaugh",
    "Taylor Spivey", "Gwen Jorgensen", "Erika Ackerlund"
]

In [28]:
from datetime import datetime

# List of sheets with known time format issues
fix_time_sheets = [
    "WC Chengdu", "WTCS Yokohama", "WC Samarkaland", "WTCS Alghero", "Huatulco WC"
]

def fix_run_time(val):
    # Only fix if value is a string and matches the pattern
    if isinstance(val, str) and val.count(":") == 2:
        h, m, s = val.split(":")
        if int(h) > 10:  # Unlikely to be a real hour value for a run
            return f"00:{h.zfill(2)}:{m.zfill(2)}"
    return val

for sheet_name, df in sheets.items():
    # Assign tier weight
    tier = 1.0 if "WTCS" in sheet_name else 0.6
    
    # Normalize column names
    df.columns = df.columns.str.replace('\n', ' ').str.strip()
    
    # Identify & rename the athlete name column
    name_cols = [c for c in df.columns if "name" in c.lower()]
    if not name_cols:
        raise ValueError(f"No athlete column found in '{sheet_name}'")
    df = df.rename(columns={name_cols[0]: "Athlete"})
    
    # Fix run time format for specific sheets
    if sheet_name in fix_time_sheets:
        for col in df.columns:
            if "run" in col.lower():
                df[col] = df[col].apply(fix_run_time)
    
    # Print the DataFrame before filtering
    #print(f"Sheet: {sheet_name} - Athletes before filtering:")
    print(df["Athlete"].tolist())
    print(df)
    
    # Add metadata columns
    df["Event"] = sheet_name
    df["Tier"] = tier
    
    # Filter to shortlist
    df = df[df["Athlete"].isin(short_list)]
    
    cleaned.append(df)




['Swim', '3rd', '27th', '18th', '19th', '23rd', nan, 'Swim Position Per Leg', '5th', '9th', '5th', '3rd']
    Individual Race                Athlete                 Unnamed: 2  \
0              Name                   Swim  Swim Split from Swim Lead   
1     Taylor Spivey                    3rd                       +:18   
2   Erika Ackerlund                   27th                       +:34   
3    Gwen Jorgensen                   18th                       +:29   
4    Morgan Pearson                   19th                       +:22   
5         John Reed                   23rd                       +:25   
6       Mixed Relay                    NaN                        NaN   
7              Name  Swim Position Per Leg         Swim Leg Time Back   
8     Taylor Spivey                    5th                       +:06   
9         John Reed                    9th                       +:11   
10  Erika Ackerlund                    5th                       +:12   
11   Morgan Pearso

In [25]:
# 4. Concatenate all into a single DataFrame
race_level_df = pd.concat(cleaned, ignore_index=True)

# 4b. Drop columns where all values are NaN
race_level_df = race_level_df.dropna(axis=1, how='all')

# 5. Save to CSV for further processing / visuals
race_level_df.to_csv("race_level.csv", index=False)

# 6. (Optional) Display first few rows
race_level_df.head(20)

,Athlete,Event,Tier,Swim Place,Swim Time,Swim Split from Swim Lead,T1 Place,T1 Time,T1 Split from Lead,Bike Analysis,T2 Place,T2 Time,T2 Split from Lead,Run Place,Run Split,Run split from run leader,Finish Time,Finish time back from lead,Finish Place
0,Reese Vannerson,WC Napier,0.6,12th,08:44:00,(+):09,12th,:34,(+):04,1 days 02:26:00,3rd,:19,(+):02,3rd,14:36:00,(+):08,50.36,(+):07,5th
1,Sullivan Middaugh,WC Napier,0.6,34th,09:05:00,(+):30,33rd,:38,(+):08,1 days 01:56:00,1st,:17,:00,11th,15:05:00,(+):37,50.58,(+):29,9th
2,John Reed,WC Napier,0.6,11th,08:43:00,(+):08,34th,:39,(+):09,1 days 02:20:00,5th,:21,(+):04,8th,15:01:00,(+):33,51.0,(+):31,10th
3,Erika Ackerlund,WC Napier,0.6,14th,09:51:00,(+):32,9th,:39,(+):02,1 days 05:26:00,10th,:21,(+):04,2nd,16:50:00,(+):05,57.05,(+):41,4th
4,Reese Vannerson,WC Chengdu,0.6,32nd,18:50:00,+1:17,11th,:41,+:02,NaN,26th,:25,+:03,1st,1 days 06:02:00,-,0 days 01:39:42,-,1st
5,Chase McQueen,WTCS Yokohama,1.0,3rd,17:50:00,+:03,36th,:52,+:07,Lead Group,3rd,:19,:00,22nd,1 days 08:05:00,+2:22,0 days 01:43:29,+2:21,13th
6,John Reed,WTCS Yokohama,1.0,26th,18:24:00,+:37,15th,:48,+:03,Chase Group,33rd,:23,+:04,8th,1 days 06:41:00,+:58,0 days 01:44:32,+3:24,21st
7,Morgan Pearson,WTCS Yokohama,1.0,32nd,18:32:00,+:45,3rd,:46,+:01,Chase Group,16th,:21,+:02,14th,1 days 07:24:00,+1:41,0 days 01:45:12,+4:04,25th
8,Gwen Jorgensen,WTCS Yokohama,1.0,21st,19:48:00,+:26,41st,:58,+:10,Came together - back to middle,31st,:26,+:05,4th,1 days 09:48:00,+:10,0 days 01:51:52,+:14,4th
9,Taylor Spivey,WTCS Yokohama,1.0,7th,19:38:00,+:16,28th,:54,+:06,Brought chase group up and rode up front,33rd,:27,+:06,10th,1 days 10:29:00,+:49,0 days 01:52:28,+:50,9th


In [ ]:
from datetime import datetime

# List of sheets with known time format issues
fix_time_sheets = [
    "WC Chengdu", "WTCS Yokohama", "WC Samarkaland", "WTCS Alghero", "Huatulco WC"
]

def fix_run_time(val):
    # Only fix if value is a string and matches the pattern
    if isinstance(val, str) and val.count(":") == 2:
        h, m, s = val.split(":")
        if int(h) > 10:  # Unlikely to be a real hour value for a run
            return f"00:{h.zfill(2)}:{m.zfill(2)}"
    return val

for sheet_name, df in sheets.items():
    # Assign tier weight
    tier = 1.0 if "WTCS" in sheet_name else 0.6
    
    # Normalize column names
    df.columns = df.columns.str.replace('\n', ' ').str.strip()
    
    # Identify & rename the athlete name column
    name_cols = [c for c in df.columns if "name" in c.lower()]
    if not name_cols:
        raise ValueError(f"No athlete column found in '{sheet_name}'")
    df = df.rename(columns={name_cols[0]: "Athlete"})
    
    # Fix run time format for specific sheets
    if sheet_name in fix_time_sheets:
        for col in df.columns:
            if "run" in col.lower():
                df[col] = df[col].apply(fix_run_time)
    
    # Print the DataFrame before filtering
    #print(f"Sheet: {sheet_name} - Athletes before filtering:")
    print(df["Athlete"].tolist())
    #print(df)
    
    # Add metadata columns
    df["Event"] = sheet_name
    df["Tier"] = tier
    
    # Filter to shortlist
    df = df[df["Athlete"].isin(short_list)]
    
    cleaned.append(df)




['Swim', '3rd', '27th', '18th', '19th', '23rd', nan, 'Swim Position Per Leg', '5th', '9th', '5th', '3rd']
['Reese Vannerson', 'Sullivan Middaugh', 'John Reed', 'Keller Norland', 'Erika Ackerlund', 'Danielle Orie']
['Reese Vannerson', 'Keller Norland']
['Chase McQueen', 'John Reed', 'Darr Smith', 'Morgan Pearson', 'Gwen Jorgensen', 'Taylor Spivey', 'Gina Sereno']
['Keller Norland', 'Reese Vannerson', 'Braxton Legg', 'Danielle Orie']
['Chase McQueen', 'John Reed', 'Seth Rider', 'Darr Smith', 'Summer Rappaport', 'Gwen Jorgensen']
['Erika Ackerlund', 'Gina Sereno', 'Naomi Ruff', 'Tamara Gorman']
